In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.arduino import Arduino_IO

import os, time, threading, multiprocessing, logging

base = BaseOverlay("base.bit")

In [ ]:
%%microblaze base.PMODA

#include <stdint.h>
#include <stdbool.h>
#include <pyprintf.h>
#include "xio_switch.h"
#include "gpio.h"
#include "timer.h"

#define SUCCESS         0
#define ERR_INVALID    -1
#define ERR_NOTREADY   -2

#define NUM_PINS        8

#define TIMER_FREQ_MHZ  100

typedef struct {
    bool inited;
    gpio gpio_dev;

} gpio_handle_t;

typedef struct {
    bool  inited;
    timer timer_dev;

} timer_handle_t;

gpio_handle_t  gpio_handles[NUM_PINS];
timer_handle_t timer_handles[NUM_PINS];

// Initialization function to cache a GPIO handle and set the
// direction; this may be more efficient for repeated IO operations

int init_gpio(unsigned int pin, unsigned int direction) {
    // pyprintf("init_gpio called for pin %u\n", pin);

    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    if ((direction != GPIO_IN) && (direction != GPIO_OUT)) {
        return ERR_INVALID;
    }

    gpio_handles[pin].gpio_dev = gpio_open(pin);
    gpio_set_direction(gpio_handles[pin].gpio_dev, direction);
    gpio_handles[pin].inited = true;

    // pyprintf("init_gpio done for pin %u\n", pin);

    return SUCCESS;
}

// Function to write to a PMOD pin

int write_gpio(unsigned int pin, unsigned int val) {
    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    if (!gpio_handles[pin].inited) {
        return ERR_NOTREADY;
    }

    if (val > 1) {
        // Technically, an error, but just limit the user input to 1
        val = 1;
    }

    gpio_write(gpio_handles[pin].gpio_dev, val);

    return SUCCESS;
}

// Function to read a PMOD pin

int read_gpio(unsigned int pin) {
    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    if (!gpio_handles[pin].inited) {
        return ERR_NOTREADY;
    }

    return gpio_read(gpio_handles[pin].gpio_dev);
}

// Initialization function to cache a timer handle and set the
// direction; this may be more efficient for repeated IO operations

int init_pwm(unsigned int pin) {
    // pyprintf("init_pwm called for pin %u\n", pin);

    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    // timer_handles[pin].timer_dev = timer_open(pin);
    timer_handles[pin].timer_dev = timer_open_device(0);
    init_io_switch();
    set_pin(pin, PWM0);

    // pyprintf("init_pwm done for pin %u\n", pin);

    return SUCCESS;
}

int start_pwm(unsigned int pin, unsigned int period_usec, unsigned int duty_cycle) {
    // pyprintf("start_pwm called for pin %u\n", pin);
    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    if ((period_usec == 0) || (period_usec >= 65536) ||
        (duty_cycle  == 0) || (duty_cycle >= 100)) {
        return ERR_INVALID;
    }

    // Convert the period input in microseconds to AXI Timer
    // clock ticks
    unsigned int period = period_usec * TIMER_FREQ_MHZ;

    // Calculate the pulse from duty_cycle
    unsigned int pulse = duty_cycle * period / 100;

    // timer_pwm_generate(timer_handles[pin].timer_dev, 2632, 666);
    timer_pwm_generate(timer_handles[pin].timer_dev, period, pulse);

    // pyprintf("start_pwm done for pin %u\n", pin);

    return SUCCESS;
}

int stop_pwm(unsigned int pin) {
    // pyprintf("stop_pwm called for pin %u\n", pin);
    if (pin >= NUM_PINS) {
        return ERR_INVALID;
    }

    timer_pwm_stop(timer_handles[pin].timer_dev);

    // pyprintf("stop_pwm done for pin %u\n", pin);

    return SUCCESS;
}

In [ ]:
%%microblaze base.PMODB

#include <stdint.h>
#include "spi.h"

static char tx_buff;
static char rx_buff;

unsigned int DATA_BYTE_SIZE = 1;
unsigned int SCLK_PIN = 0;
unsigned int MISO_PIN = 1;
unsigned int MOSI_PIN = 2;
unsigned int CS_PIN = 3;

spi spi_dev;

// Initialize SPI device (PYNQ will be master)
int spi_init(void) {
    spi_dev = spi_open(SCLK_PIN, MISO_PIN, MOSI_PIN, CS_PIN);
    return 0;
}

// Read data over SPI
char spi_read_data() {
    spi_transfer(spi_dev, &tx_buff, &rx_buff, DATA_BYTE_SIZE);
    return rx_buff;
}

In [ ]:
# Helper functions
def _BV(bit):
    return (1 << bit)

# GPIO direction constants (referenced from MicroBlaze gpio.h)
# See https://github.com/Xilinx/PYNQ/blob/master/boards/sw_repo/pynqmb/src/gpio.h
GPIO_OUT = 0
GPIO_IN  = 1

# Following constants were referenced from AdaFruit motor shield v1 library
# See https://github.com/adafruit/Adafruit-Motor-Shield-library

# Motor -> Arduino header pin mapping
MOTOR_PIN_LATCH  = 12
MOTOR_PIN_CLK    = 4
MOTOR_PIN_ENABLE = 7
MOTOR_PIN_DATA   = 8

# Bit positions in the 74HCT595 shift register output
MOTOR1_A = 2
MOTOR1_B = 3
MOTOR2_A = 1
MOTOR2_B = 4
MOTOR4_A = 0
MOTOR4_B = 6
MOTOR3_A = 5
MOTOR3_B = 7

# Constants that the user passes in to the motor calls
FORWARD  = 1
BACKWARD = 2
BRAKE    = 3
RELEASE  = 4

# Constants that encode the joystick D-pad user input
# This corresponds to map_to_direction() in joystick_controller.ino
DPAD_LEFT           = 1
DPAD_RIGHT          = 2
DPAD_FORWARD        = 3
DPAD_BACKWARD       = 4
DPAD_FORWARD_LEFT   = 5
DPAD_FORWARD_RIGHT  = 6
DPAD_BACKWARD_LEFT  = 7
DPAD_BACKWARD_RIGHT = 8
DPAD_NEUTRAL        = 0

cmd_map = [
    "neutral",
    "left",
    "right",
    "forward",
    "backward",
    "forward_left",
    "forward_right",
    "backward_left",
    "backward_right",
]
    

# PWM rate for DC motors
DC_MOTOR_PWM_RATE = _BV(1) # MOTOR34_8KHZ

latch_state = 0

In [ ]:
class MotorController:
    def __init__(self):
        self.TimerInitalized = False

    def enable(self):
        global latch_state
        self.motor_pin_latch = Arduino_IO(base.ARDUINO, MOTOR_PIN_LATCH, 'out')
        self.motor_pin_enable = Arduino_IO(base.ARDUINO, MOTOR_PIN_ENABLE, 'out')
        self.motor_pin_data = Arduino_IO(base.ARDUINO, MOTOR_PIN_DATA, 'out')
        self.motor_pin_clk = Arduino_IO(base.ARDUINO, MOTOR_PIN_CLK, 'out')

        latch_state = 0
        self.latch_tx() # Reset

        self.motor_pin_enable.write(0)

    def latch_tx(self):
        global latch_state

        self.motor_pin_latch.write(0)
        self.motor_pin_data.write(0)

        for i in range(0, 8):
            self.motor_pin_clk.write(0)

            if (latch_state & _BV(7-i)):
                self.motor_pin_data.write(1)
            else:
                self.motor_pin_data.write(0)

            self.motor_pin_clk.write(1)

        self.motor_pin_latch.write(1)

class DCMotor:
    def __init__(self, motornum, MotorController, freq = DC_MOTOR_PWM_RATE):
        global latch_state

        self.motornum = motornum
        self.pwmfreq = freq
        self.MotorController = MotorController

        MotorController.enable()
        if (motornum == 1):
            self.pwm = Arduino_IO(base.ARDUINO, 11, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR1_A) & ~_BV(MOTOR1_B)
            MotorController.latch_tx()

        elif (motornum == 2):
            self.pwm = Arduino_IO(base.ARDUINO, 3, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR2_A) & ~_BV(MOTOR2_B)
            MotorController.latch_tx()

        elif (motornum == 3):
            self.pwm = Arduino_IO(base.ARDUINO, 6, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR3_A) & ~_BV(MOTOR3_B)
            MotorController.latch_tx()

        elif (motornum == 4):
            self.pwm = Arduino_IO(base.ARDUINO, 5, 'out')
            # set both motor pins to 0
            latch_state &= ~_BV(MOTOR4_A) & ~_BV(MOTOR4_B)
            MotorController.latch_tx()

    def run(self, cmd):
        global latch_state

        if (self.motornum == 1):
            a = MOTOR1_A
            b = MOTOR1_B
        elif (self.motornum == 2):
            a = MOTOR2_A
            b = MOTOR2_B
        elif (self.motornum == 3):
            a = MOTOR3_A
            b = MOTOR3_B
        elif (self.motornum == 4):
            a = MOTOR4_A
            b = MOTOR4_B

        if (cmd == FORWARD):
            latch_state |= _BV(a)
            latch_state &= ~_BV(b)
            self.MotorController.latch_tx()

        elif (cmd == BACKWARD):
            latch_state &= ~_BV(a);
            latch_state |= _BV(b);
            self.MotorController.latch_tx()

        elif (cmd == RELEASE):
            # A and B both low
            latch_state &= ~_BV(a)
            latch_state &= ~_BV(b); 
            self.MotorController.latch_tx()

    def pwm_t(self, speed):
        t = threading.Thread(target=worker_t, args=(forks, i, btn_press_event))

    def set_speed(self, speed):
        return

In [ ]:
class RC_Car:
    def __init__(self, weapons = True, log_level = logging.INFO):
        # Logging
        self.logger = logging.getLogger("PYNQ-Tag")
        self.logger.setLevel(log_level)
        self.logfile_handler = logging.FileHandler("PYNQ-Tag.log", mode="w")

        self.logfile_handler.setFormatter(
            logging.Formatter(
                "%(asctime)s.%(msecs)03d - %(levelname)s - %(message)s",
                datefmt="%Y-%m-%d %H:%M:%S"))
        self.logger.addHandler(self.logfile_handler)

        self.logger.info(f"Beginning program...")
        self.logger.debug(f"Logging initialized")

        # Motors
        self.logger.debug(f"Initializing motors")
        # TODO: Config file?
        # Motor 0 is front-left (M1)
        # Motor 1 is back-left (M2)
        # Motor 2 is back-right (M3)
        # Motor 3 is front-right (M4)
        self.motor_controller = MotorController()
        self.motors = []
        for i in range(1, 5):
            self.motors.append(DCMotor(i, self.motor_controller))
        self.motor_fl = self.motors[0]
        self.motor_fr = self.motors[3]
        self.motor_bl = self.motors[1]
        self.motor_br = self.motors[2]
        self.logger.debug(f"Motors initialized")

        # Weapons
        self.weapons = weapons
        if weapons:
            self.logger.debug(f"Initializing laser and IR receiver")
        else:
            self.logger.debug(f"Skipping laser and IR receiver initialization")

        self.pmodA_lock = threading.Lock()
        self.laser = self.Laser(parent_class = self)
        self.ir_receiver = self.IR_Receiver(parent_class = self)
        if weapons:
            self.logger.debug(f"Laser and IR receiver initialized")

    def fire_laser(self):
        self.laser.shoot_event.set()

    def stop(self):
        self.logger.info(f"Stopping program...")
        self.logfile_handler.close()
        for motor in self.motors:
            motor.run(RELEASE)

    def move(self, cmd):
        global cmd_map
        self.logger.debug(f"Received command: {cmd_map[cmd]}")
        if (cmd == DPAD_FORWARD):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_BACKWARD):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_LEFT):
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_RIGHT):
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_FORWARD_LEFT):
            #self.motors[0].set_speed(1)
            #self.motors[2].set_speed(1)
            self.motor_fl.run(RELEASE)
            self.motor_fr.run(FORWARD)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(FORWARD)

        elif (cmd == DPAD_FORWARD_RIGHT):
            #self.motors[1].set_speed(1)
            #self.motors[3].set_speed(1)
            self.motor_fl.run(FORWARD)
            self.motor_fr.run(RELEASE)
            self.motor_bl.run(FORWARD)
            self.motor_br.run(RELEASE)

        elif (cmd == DPAD_BACKWARD_LEFT):
            #self.motors[0].set_speed(1)
            #self.motors[2].set_speed(1)
            self.motor_fl.run(RELEASE)
            self.motor_fr.run(BACKWARD)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(BACKWARD)

        elif (cmd == DPAD_BACKWARD_RIGHT):
            #self.motors[1].set_speed(1)
            #self.motors[3].set_speed(1)
            self.motor_fl.run(BACKWARD)
            self.motor_fr.run(RELEASE)
            self.motor_bl.run(BACKWARD)
            self.motor_br.run(RELEASE)

        elif (cmd == DPAD_NEUTRAL):
            self.motor_fl.run(RELEASE)
            self.motor_fr.run(RELEASE)
            self.motor_bl.run(RELEASE)
            self.motor_br.run(RELEASE)

    class Laser:
        def __init__(self, parent_class, pin = 7):
            # IR transmitter "laser" is on PMODA and connected to
            # pin 7

            self.logger = parent_class.logger
            self.enable = parent_class.weapons
            if not self.enable:
                self.logger.info(f"Laser not initialized")
                return

            err = init_pwm(pin)
            if (err != 0):
                self.logger.error(
                    f"init_pwm failed for pin {pin}, err: {err}")
            else:
                self.logger.debug(
                    f"init_pwm successful for pin {pin}")

            self.pwm_period_usec = 26 # 26 usec is about 38.4 KHz
            self.pwm_duty_cycle = 50
            self.pwm_pin = pin
            self.laser_pulse_duration_msec = 250 # 250 msec pulse

            self.shoot_event = threading.Event()
            self.shoot_thread = threading.Thread(
                target = self.shoot_t,
                args = (parent_class.pmodA_lock, self.shoot_event))

            self.shoot_thread.start()

            self.logger.info(f"Initialized laser shoot thread")
            self.logger.debug(f"Laser signal pin: {self.pwm_pin}")
            self.logger.debug(
                f"PWM frequency: {1000.0 / self.pwm_period_usec:.2f} KHz")
            self.logger.debug(
                f"Laser pulse duration: {self.laser_pulse_duration_msec} milliseconds")

        def shoot_t(self, lock, shoot_event):
            while True:
                # Wait until shoot_event is set
                if not shoot_event.is_set():
                    continue

                self._shoot(lock)
                shoot_event.clear()

        def _shoot(self, lock):
            if not self.enable:
                self.logger.debug(f"Laser could not be shot - not initialized!")
                return

            # Shoot a quarter second pulse "laser"
            lock.acquire()
            err = start_pwm(
                self.pwm_pin,
                self.pwm_period_usec,
                self.pwm_duty_cycle)
            lock.release()

            if (err != 0):
                self.logger.error(
                    f"start_pwm failed for pin {self.pwm_pin}, err: {err}")
            else:
                self.logger.debug(
                    f"start_pwm successful for pin {self.pwm_pin}")

            time.sleep(self.laser_pulse_duration_msec / 1000)

            lock.acquire()
            err = stop_pwm(self.pwm_pin)
            lock.release()

            if (err != 0):
                self.logger.error(
                    f"stop_pwm failed for pin {self.pwm_pin}, err: {err}")
            else:
                self.logger.debug(
                    f"stop_pwm successful for pin {self.pwm_pin}")

            self.logger.info(f"Laser shot!")

    class IR_Receiver():
        def __init__(self, parent_class, pin = 3):
            # IR receiver is on PMODA and connected to pin 3
            self.enable = parent_class.weapons
            self.logger = parent_class.logger
            if not self.enable:
                self.logger.info(f"IR receiver not initialized")
                return

            err = init_gpio(pin, GPIO_IN)
            if (err != 0):
                self.logger.error(
                    f"init_gpio failed for pin {pin}, err: {err}")
            else:
                self.logger.debug(
                    f"init_gpio successful for pin {pin}")

            self.ir_pin = pin

            # Start a process which indicates a hit
            self.ir_rx_thread = threading.Thread(
                target = self.notify_hit_t,
                args = (parent_class.pmodA_lock, ))

            #self.proc = multiprocessing.Process(
            #                target=self.notify_hit_p, args=())
            #self.proc.start()
            #os.system("taskset -p {} {}".format(0x2, self.proc.pid))

            self.ir_rx_thread.start()

            self.logger.info(f"Initialized IR Receiver thread")
            self.logger.debug(f"IR receiver signal pin: {self.ir_pin}")

        def notify_hit_t(self, lock):
            if not self.enable:
                return

            lock.acquire()
            prev_read = read_gpio(self.ir_pin)
            lock.release()
            while True:
                lock.acquire()
                curr_read = read_gpio(self.ir_pin)
                lock.release()
                if curr_read == 0 and curr_read != prev_read:
                    # TODO: This will send a kill signal?
                    self.logger.info("Hit!")
                    print("Hit!")
                prev_read = curr_read
                time.sleep(0.01)


In [ ]:
car = RC_Car(True, logging.DEBUG)

In [ ]:
for _ in range(0, 5):
    car.fire_laser()
    time.sleep(1)

In [ ]:
car.move(DPAD_FORWARD)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_BACKWARD)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_LEFT)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_RIGHT)
time.sleep(0.1)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_FORWARD_RIGHT)
time.sleep(0.25)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_FORWARD_LEFT)
time.sleep(0.25)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_BACKWARD_LEFT)
time.sleep(0.25)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.move(DPAD_BACKWARD_RIGHT)
time.sleep(0.25)
car.move(DPAD_NEUTRAL)
time.sleep(0.1)
car.stop()

In [ ]:
spi_init()

prev = -1

while True:
    cmd = spi_read_data()
    # We only expect a byte of data, but the SPI read is giving
    # 0xFFFF_FFFF sometimes
    if cmd > 255:
        continue

    if cmd != prev:
        print(f"Command received over SPI: {cmd}")
        prev = cmd